# Registrar los productos MEDI (INEGI) en el catálogo STAC (`data/stac/inegi/`)

Paso **S4** de `PLAN-SOCIOECONOMICOS.md`. Abre el catálogo `atlas` existente
(creado por la libreta 003), reemplaza la colección `inegi` si ya estaba y
registra tres items con los productos de las libretas 005 y 006:

| item | asset `data` | qué es |
|---|---|---|
| `medi-ageb-2020` | `medi_ageb_2020.parquet` | GeoParquet, 64 313 AGEB |
| `medi-mun-2020`  | `medi_mun_2020.parquet`  | GeoParquet, 2 469 municipios |
| `medi-grid-2020` | `medi_ageb_2020_grid.parquet` | tabla puente AGEB → celda UTCI |

Cada item usa la extensión **table** (columnas con tipo y descripción) y una
propiedad propia `atlas:indicators` con los indicadores que la app debe
ofrecer (id, etiqueta, columna, peso MEDI, numerador y denominador). La app
descubre los años y los indicadores desde aquí, nunca con rutas sueltas.
El STAC **cataloga, no duplica**: los assets apuntan por ruta relativa a
`data/derived/`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import pystac
import geopandas as gpd

ROOT = Path("..").resolve()
STAC_DIR = ROOT / "data" / "stac"
DERIVED = ROOT / "data" / "derived" / "INEGI" / "2020"

YEAR = 2020
PARQUET_MEDIA_TYPE = "application/vnd.apache.parquet"
TABLE_EXT = "https://stac-extensions.github.io/table/v1.2.0/schema.json"
LICENSE_URL = "https://www.inegi.org.mx/inegi/terminos.html"

PRODUCTOS = {
    "ageb": DERIVED / "medi_ageb_2020.parquet",
    "mun": DERIVED / "medi_mun_2020.parquet",
    "grid": DERIVED / "medi_ageb_2020_grid.parquet",
}
for p in PRODUCTOS.values():
    assert p.exists(), f"falta {p}: corre 005 y 006"

# Indicadores que expone la app (mismo orden y pesos del diseño MEDI).
INDICATORS = [
    {
        "id": "sin_elec",
        "label": "Viviendas sin electricidad (%)",
        "column": "p_sin_elec",
        "upper_bound_column": "p_sin_elec_sup",
        "flag_column": "flag_elec",
        "weight_medi": 0.24,
        "numerator": "vph_s_elec",
        "denominator": "vivparh_cv",
    },
    {
        "id": "sin_refri",
        "label": "Viviendas sin refrigerador (%)",
        "column": "p_sin_refri",
        "upper_bound_column": None,
        "flag_column": "flag_refri",
        "weight_medi": 0.21,
        "numerator": "vph_sin_refri",
        "denominator": "vivparh_cv",
    },
]

# Descripciones de columnas para la extensión table (las que no estén aquí
# se registran sólo con nombre y tipo).
DESCRIPCIONES = {
    "cvegeo": "Clave geoestadística INEGI (13 = AGEB urbana, 5 = municipio)",
    "cve_ent": "Clave de entidad", "cve_mun": "Clave de municipio", "cve_loc": "Clave de localidad",
    "cve_ageb": "Clave de AGEB", "nom_ent": "Entidad", "nom_mun": "Municipio", "nom_loc": "Localidad",
    "ambito_mgn": "Capa del marco de donde vino el polígono (urbana | rural)",
    "pobtot": "Población total",
    "vivpar_hab": "Viviendas particulares habitadas",
    "vivparh_cv": "Viviendas particulares habitadas con características captadas (denominador de los VPH_*)",
    "vph_c_elec": "Viviendas con energía eléctrica", "vph_s_elec": "Viviendas sin energía eléctrica",
    "vph_refri": "Viviendas con refrigerador", "vph_sin_refri": "Viviendas sin refrigerador (vivparh_cv − vph_refri)",
    "vph_sinrtv": "Viviendas sin radio ni televisor", "vph_sinltc": "Viviendas sin línea telefónica ni celular",
    "p_sin_elec": "% viviendas sin electricidad (nulo si censurado o sin viviendas)",
    "p_sin_elec_sup": "Cota superior de p_sin_elec cuando el numerador está censurado (< 3 viviendas)",
    "p_sin_refri": "% viviendas sin refrigerador",
    "flag_elec": "ok | censurado | sin_viviendas", "flag_refri": "ok | censurado | sin_viviendas",
    "n_ageb": "AGEB urbanas del municipio en el ITER", "pob_ageb_urb": "Población en AGEB urbanas",
    "cent_lon": "Longitud del centroide (calculado en EPSG:6372)", "cent_lat": "Latitud del centroide",
    "lat_c": "Latitud del centro de la celda UTCI de 0.25° que contiene el centroide",
    "lon_c": "Longitud del centro de la celda UTCI",
    "geometry": "Polígono del Marco Geoestadístico 2020, EPSG:4326",
}

## 1. Metadatos de cada parquet (esquema, filas, bbox) sin cargarlo entero

In [2]:
def parquet_meta(path: Path) -> dict:
    f = pq.ParquetFile(path)
    schema = f.schema_arrow
    cols = []
    for field in schema:
        if field.name in ("bbox",):          # columna auxiliar de GeoParquet (covering bbox)
            continue
        tipo = "geometry" if field.name == "geometry" else str(field.type)
        col = {"name": field.name, "type": tipo}
        if field.name in DESCRIPCIONES:
            col["description"] = DESCRIPCIONES[field.name]
        cols.append(col)
    meta = {"columns": cols, "rows": f.metadata.num_rows}
    geo = f.schema_arrow.metadata.get(b"geo") if f.schema_arrow.metadata else None
    if geo:
        import json
        g = json.loads(geo)
        meta["bbox"] = g["columns"]["geometry"]["bbox"]
        meta["geometry"] = "geometry"
    return meta


metas = {k: parquet_meta(p) for k, p in PRODUCTOS.items()}
for k, m in metas.items():
    print(f"{k:5s} filas={m['rows']:6d} cols={len(m['columns'])} bbox={m.get('bbox')}")

# La tabla puente no tiene geometría: hereda el bbox de la AGEB.
bbox_ageb = [round(v, 4) for v in metas["ageb"]["bbox"]]
bbox_mun = [round(v, 4) for v in metas["mun"]["bbox"]]
metas["grid"]["bbox"] = bbox_ageb

ageb  filas= 64313 cols=26 bbox=[-117.12420240005815, 14.58621095010732, -86.71093514017757, 32.71845757092842]
mun   filas=  2469 cols=24 bbox=[-118.3651143522829, 14.532098369766466, -86.71040527988355, 32.71865357039013]
grid  filas= 64313 cols=11 bbox=None


## 2. Colección `inegi` dentro del catálogo existente

In [3]:
catalog = pystac.Catalog.from_file(str(STAC_DIR / "catalog.json"))
if catalog.get_child("inegi") is not None:
    catalog.remove_child("inegi")

# Censo 2020: levantamiento 2 – 27 de marzo, fecha de referencia 15 de marzo.
T0 = datetime(2020, 3, 2, tzinfo=timezone.utc)
T1 = datetime(2020, 3, 27, tzinfo=timezone.utc)
T_REF = datetime(2020, 3, 15, tzinfo=timezone.utc)

collection = pystac.Collection(
    id="inegi",
    title="INEGI — indicadores socioeconómicos (Censo 2020, MEDI)",
    description=(
        "Indicadores de carencia energética del diseño MEDI a partir del Censo de "
        "Población y Vivienda 2020 (INEGI): viviendas sin electricidad y sin "
        "refrigerador, por AGEB urbana y por municipio, sobre el Marco "
        "Geoestadístico 2020. Incluye la tabla puente AGEB → celda de 0.25° de la "
        "malla UTCI para cruces con el estrés térmico."
    ),
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([bbox_mun]),
        temporal=pystac.TemporalExtent([[T0, T1]]),
    ),
    license="other",
    keywords=["INEGI", "Censo 2020", "AGEB", "municipio", "MEDI", "pobreza energética", "México"],
    providers=[
        pystac.Provider(
            name="INEGI",
            description="Censo de Población y Vivienda 2020: Principales resultados por AGEB y manzana urbana; Marco Geoestadístico 2020",
            roles=[pystac.ProviderRole.PRODUCER, pystac.ProviderRole.LICENSOR],
            url="https://www.inegi.org.mx/programas/ccpv/2020/",
        ),
        pystac.Provider(
            name="IER-UNAM (atlas)",
            roles=[pystac.ProviderRole.PROCESSOR],
        ),
    ],
)
collection.add_link(pystac.Link(rel="license", target=LICENSE_URL, title="Términos de Libre Uso de la Información del INEGI"))
catalog.add_child(collection)

<Link rel=child target=<Collection id=inegi>>

## 3. Items (uno por producto) con extensión table y `atlas:indicators`

In [4]:
def bbox_geometry(b):
    return {"type": "Polygon", "coordinates": [[[b[0], b[1]], [b[2], b[1]], [b[2], b[3]], [b[0], b[3]], [b[0], b[1]]]]}


TITULOS = {
    "ageb": "MEDI por AGEB urbana, Censo 2020 (GeoParquet, EPSG:4326)",
    "mun": "MEDI por municipio, Censo 2020 (GeoParquet, EPSG:4326)",
    "grid": "Tabla puente AGEB → celda UTCI 0.25° con conteos MEDI (Parquet)",
}
NIVEL = {"ageb": "ageb", "mun": "municipio", "grid": "ageb_grid"}

for k, path in PRODUCTOS.items():
    m = metas[k]
    props = {
        "start_datetime": T0.isoformat(),
        "end_datetime": T1.isoformat(),
        "atlas:level": NIVEL[k],
        "atlas:census_year": YEAR,
        "atlas:crs": "EPSG:4326" if k != "grid" else None,
        "atlas:grid_res_deg": 0.25 if k == "grid" else None,
        "atlas:indicators": INDICATORS,
        "atlas:source": [
            "INEGI. Censo de Población y Vivienda 2020. Principales resultados por AGEB y manzana urbana (datos abiertos, 32 archivos)",
            "INEGI. Marco Geoestadístico. Censo de Población y Vivienda 2020 (mg_2020_integrado)",
        ],
        "atlas:notebook": {"ageb": "notebooks/005_MEDI_ageb.ipynb", "mun": "notebooks/006_MEDI_mun.ipynb", "grid": "notebooks/006_MEDI_mun.ipynb"}[k],
        "table:columns": m["columns"],
        "table:row_count": m["rows"],
    }
    if "geometry" in m:
        props["table:primary_geometry"] = "geometry"
    props = {a: b for a, b in props.items() if b is not None}

    item = pystac.Item(
        id=f"medi-{k}-{YEAR}",
        geometry=bbox_geometry(m["bbox"]),
        bbox=m["bbox"],
        datetime=T_REF,
        properties=props,
        stac_extensions=[TABLE_EXT],
    )
    item.add_asset(
        "data",
        pystac.Asset(
            href=str(path),
            media_type=PARQUET_MEDIA_TYPE,
            title=TITULOS[k],
            roles=["data"],
        ),
    )
    collection.add_item(item)
    print("item", item.id, "filas", m["rows"])

item medi-ageb-2020 filas 64313
item medi-mun-2020 filas 2469
item medi-grid-2020 filas 64313


## 4. Guardar, validar y comprobar la ida y vuelta

In [5]:
catalog.normalize_hrefs(str(STAC_DIR))
catalog.make_all_asset_hrefs_relative()
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
print("Guardado en", STAC_DIR)
catalog.describe()

Guardado en /Users/gbv/atlas/data/stac
* <Catalog id=atlas>
    * <Collection id=utci>
      * <Item id=utci-hourly-2022>
      * <Item id=utci-levels-2022>
      * <Item id=utci-hourly-2023>
      * <Item id=utci-levels-2023>
    * <Collection id=inegi>
      * <Item id=medi-ageb-2020>
      * <Item id=medi-mun-2020>
      * <Item id=medi-grid-2020>


In [6]:
for obj in [catalog, collection, *collection.get_items()]:
    obj.validate()
    print("OK", obj.id)
# La colección utci debe seguir intacta
utci = catalog.get_child("utci")
assert utci is not None and len(list(utci.get_items())) >= 2, "la colección utci se perdió"
print("utci intacta:", [it.id for it in utci.get_items()])

OK atlas
OK inegi
OK medi-ageb-2020
OK medi-mun-2020
OK medi-grid-2020
utci intacta: ['utci-hourly-2022', 'utci-levels-2022', 'utci-hourly-2023', 'utci-levels-2023']


In [7]:
cat = pystac.Catalog.from_file(str(STAC_DIR / "catalog.json"))
col = cat.get_child("inegi")
for it in col.get_items():
    href = Path(it.assets["data"].get_absolute_href())
    assert href.exists(), href
    if "table:primary_geometry" in it.properties:
        df = gpd.read_parquet(href, columns=["cvegeo", "p_sin_refri", "geometry"])
        print(it.id, "→", href.relative_to(ROOT), "|", len(df), "filas, CRS", df.crs.to_epsg())
    else:
        df = pd.read_parquet(href)
        print(it.id, "→", href.relative_to(ROOT), "|", len(df), "filas (sin geometría)")
    assert len(df) == it.properties["table:row_count"]
print("indicadores publicados:", [i["id"] for i in col.get_item(f"medi-ageb-{YEAR}").properties["atlas:indicators"]])

medi-ageb-2020 → data/derived/INEGI/2020/medi_ageb_2020.parquet | 64313 filas, CRS 4326
medi-mun-2020 → data/derived/INEGI/2020/medi_mun_2020.parquet | 2469 filas, CRS 4326
medi-grid-2020 → data/derived/INEGI/2020/medi_ageb_2020_grid.parquet | 64313 filas (sin geometría)
indicadores publicados: ['sin_elec', 'sin_refri']
